In [1]:
%load_ext autoreload
%autoreload 2

import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import joblib
warnings.filterwarnings('ignore')

sys.path.append('../../scripts')
from data_pipeline import load_and_prepare_data, test_train_split
from feature_engineering import build_state_vector
import plots, xgb_scripts, evaluate

In [6]:
df_train, df_test = test_train_split(
      data_folder="../../data/04-03-24", 
      method="bin_and_split"
)

In [7]:
# 1) Build state vectors exactly as you do now (but keep them as Python lists)
state_vectors, valid_cycles = build_state_vector(df_train)  # same args you use in build_model_input

# 2) Check per-row lengths
lengths = [len(v) if hasattr(v, '__len__') else None for v in state_vectors]
unique_lengths = sorted(set(lengths))
print("Unique vector lengths:", unique_lengths)

# 3) Find offending indices/cycles that aren't expected length (74 here)
expected = max(set(lengths), key=lengths.count)  # the modal length (should be 74)
bad_idx = [i for i, L in enumerate(lengths) if L != expected]
print("Bad indices:", bad_idx)
print("Bad cycles:", [valid_cycles[i] for i in bad_idx])

# 4) Look for non-scalars inside those vectors
def non_scalar_positions(vec):
    pos = []
    for j, val in enumerate(vec):
        # scalars: int/float/np.floating/np.nan
        if isinstance(val, (list, tuple, pd.Series, np.ndarray)):
            pos.append((j, type(val).__name__))
    return pos

for i in bad_idx[:5]:
    print("Inspecting cycle:", valid_cycles[i])
    print("non-scalar positions:", non_scalar_positions(state_vectors[i]))

# 5) If you suspect duplicate rows in Ns==1 only for those cycles:
# df_ns1 = df[(df['Ns'] == 1) & (df['cycle number'].isin([valid_cycles[i] for i in bad_idx]))]
# dups = (df_ns1.groupby(['cycle number','freq/Hz']).size()
#         .reset_index(name='count')).query("count > 1")
# print("Duplicates (if any) in the bad cycles:\n", dups.head(20))


NON-SCALAR @ channel=A1 cycle=1.0 freq=0.254 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=0.34 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=0.456 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=0.612 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=0.822 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=1.1 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=1.48 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=1.99 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=2.66 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=3.57 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=4.8 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=6.43 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=8.64 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=11.6 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=15.5 re_len=8 im_len=8
NON-SCALAR @ channel=A1 cycle=1.0 freq=20.9 re_len=8 im_len=8
NON-SC

In [5]:
d = df_train.query("`freq/Hz` > 0.2 and `freq/Hz` <= 20000 and Ns == 1").copy()
d["f"] = d["freq/Hz"].round(3)

dup_counts = (d.groupby(["channel","cycle number","f"])
                .size().reset_index(name="count"))
dups = dup_counts.query("count > 1")
print(dups.head())  # now you’ll see exactly which file (channel) is responsible


Empty DataFrame
Columns: [channel, cycle number, f, count]
Index: []
